#  Análise Exploratória de Dados (EDA) — Assistência Estudantil UFPB (Campus I)

##  1. Visão Geral e Objetivos do Projeto
Este notebook contempla a **Análise Exploratória de Dados (EDA)** sobre a base consolidada da **Pró-Reitoria de Assistência e Promoção ao Estudante (PRAPE)** da UFPB (Campus I - João Pessoa).

###  Objetivos Principais:
* **Mapeamento Demográfico:** Avaliar o perfil socioeconômico e de raça/cor dos discentes matriculados.
* **Cobertura da Assistência:** Analisar a penetração das modalidades de apoio (Social, Alimentação, Moradia, Transporte, etc.).
* **Avaliação de Políticas Afirmativas:** Medir a proporção de estudantes ingressantes via reserva de vagas (Lei de Cotas).
* **Cruzamento de Vulnerabilidades:** Correlacionar raça/cor, turno do curso e concessão de auxílios para identificar gargalos de permanência.

---

##  2. Configuração do Ambiente e Carga dos Dados

nesta etapa, importamos as bibliotecas necessárias, definimos os caminhos relativos para os arquivos processados e carregamos a **Tabela Fato** juntamente com as **Tabelas de Dimensão**.

In [17]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Configurações do Pandas e Estilo Visual
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: "%.2f" % x)
sns.set_theme(style="whitegrid")

# Mapeamento de Diretórios Relativos
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PATH_FATO = BASE_DIR / "data" / "processed" / "Fato" / "fato_assistencia.csv"
PATH_DIM_CURSO = (
    BASE_DIR / "data" / "processed" / "Dimensões" / "dim_curso_ufpb_campus_1.csv"
)
PATH_DIM_RACA = BASE_DIR / "data" / "processed" / "Dimensões" / "dim_raca.csv"
PATH_DIM_TURNO = BASE_DIR / "data" / "processed" / "Dimensões" / "dim_turno.csv"

# Carregamento e Enriquecimento dos Dados
df_fato = pd.read_csv(PATH_FATO, sep=";")

if PATH_DIM_CURSO.exists():
    df_curso = pd.read_csv(PATH_DIM_CURSO, sep=";")
    df_fato = df_fato.merge(
        df_curso[["CO_CURSO", "NO_CURSO"]], on="CO_CURSO", how="left"
    )

if PATH_DIM_RACA.exists():
    df_raca = pd.read_csv(PATH_DIM_RACA, sep=";")
    col_desc_raca = next(
        (c for c in ["DESCRICAO", "DS_RACA", "NO_RACA"] if c in df_raca.columns),
        df_raca.columns[-1],
    )
    df_fato = df_fato.merge(
        df_raca[["ID_RACA", col_desc_raca]], on="ID_RACA", how="left"
    )
    df_fato.rename(columns={col_desc_raca: "RACA_DESCRICAO"}, inplace=True)

if PATH_DIM_TURNO.exists():
    df_turno = pd.read_csv(PATH_DIM_TURNO, sep=";")
    col_desc_turno = next(
        (
            c
            for c in ["DESCRICAO", "DS_TURNO", "NO_TURNO"]
            if c in df_turno.columns
        ),
        df_turno.columns[-1],
    )
    df_fato = df_fato.merge(
        df_turno[["ID_TURNO", col_desc_turno]], on="ID_TURNO", how="left"
    )
    df_fato.rename(columns={col_desc_turno: "TURNO_DESCRICAO"}, inplace=True)

print(
    f"Carga concluída com sucesso! Total de registros agrupados: {len(df_fato):,}"
)

Carga concluída com sucesso! Total de registros agrupados: 2,468


---

##  3. Métricas Globais (Macro KPI Analysis)

Consolidação do volume total de alunos analisados na amostra e distribuição absoluta de cotistas e beneficiários.

In [19]:
total_alunos = df_fato["TOTAL_ALUNOS"].sum()
total_cotistas = df_fato[df_fato["IN_RESERVA_VAGAS"] == 1][
    "TOTAL_ALUNOS"
].sum()
total_apoio_social = df_fato[df_fato["IN_APOIO_SOCIAL"] == 1][
    "TOTAL_ALUNOS"
].sum()
total_alimentacao = df_fato[df_fato["IN_APOIO_ALIMENTACAO"] == 1][
    "TOTAL_ALUNOS"
].sum()

print("=" * 50)
print(" METRICAS GLOBAIS - UFPB CAMPUS I")
print("=" * 50)
print(f"Total de Alunos Analisados: {total_alunos:,}")
print(
    f"Alunos Ingressantes por Cotas: {total_cotistas:,} ({(total_cotistas/total_alunos*100):.2f}%)"
)
print(
    f"Alunos Atendidos com Apoio Social: {total_apoio_social:,} ({(total_apoio_social/total_alunos*100):.2f}%)"
)
print(
    f"Alunos Atendidos com Alimentação (RU): {total_alimentacao:,} ({(total_alimentacao/total_alunos*100):.2f}%)"
)

 METRICAS GLOBAIS - UFPB CAMPUS I
Total de Alunos Analisados: 31,210
Alunos Ingressantes por Cotas: 13,303 (42.62%)
Alunos Atendidos com Apoio Social: 2,637 (8.45%)
Alunos Atendidos com Alimentação (RU): 1,419 (4.55%)


---

##  4. Ranking Cursos e Concentração Discente

Mapeamento do volume discente por curso no Campus I da UFPB.

In [20]:
top_cursos = (
    df_fato.groupby("NO_CURSO")["TOTAL_ALUNOS"]
    .sum()
    .reset_index()
    .sort_values(by="TOTAL_ALUNOS", ascending=False)
)

top_cursos["% REPR"] = (
    top_cursos["TOTAL_ALUNOS"] / top_cursos["TOTAL_ALUNOS"].sum() * 100
).round(2)

print("--- TOP 10 CURSOS COM MAIOR VOLUME DE ALUNOS ---")
display(top_cursos.head(10))

--- TOP 10 CURSOS COM MAIOR VOLUME DE ALUNOS ---


,NO_CURSO,TOTAL_ALUNOS,% REPR
61,PEDAGOGIA,1694,5.43
15,CIÊNCIAS CONTÁBEIS,1319,4.23
22,DIREITO,1041,3.34
52,LETRAS - LÍNGUA PORTUGUESA,981,3.14
23,EDUCAÇÃO FÍSICA,780,2.50
57,MEDICINA,779,2.50
50,LETRAS - INGLÊS,747,2.39
55,MATEMÁTICA,675,2.16
0,ADMINISTRAÇÃO,674,2.16
14,CIÊNCIAS BIOLÓGICAS,664,2.13


---

##  5. Análise Cruzada de Ações Afirmativas e Raça/Cor

Investigação da taxa de adesão à reserva de vagas (Lei de Cotas) entre diferentes autodeclarações raciais.

In [21]:
df_cota_raca = (
    df_fato.groupby(["RACA_DESCRICAO", "IN_RESERVA_VAGAS"])["TOTAL_ALUNOS"]
    .sum()
    .unstack(fill_value=0)
)
df_cota_raca.rename(
    columns={0: "AMPLA_CONCORRENCIA", 1: "RESERVA_VAGAS"}, inplace=True
)

df_cota_raca["TOTAL"] = (
    df_cota_raca["AMPLA_CONCORRENCIA"] + df_cota_raca["RESERVA_VAGAS"]
)
df_cota_raca["% COTISTAS"] = (
    df_cota_raca["RESERVA_VAGAS"] / df_cota_raca["TOTAL"] * 100
).round(2)

print("--- DEMANDA DE RESERVA DE VAGAS POR RAÇA/COR ---")
display(df_cota_raca.sort_values(by="TOTAL", ascending=False))

--- DEMANDA DE RESERVA DE VAGAS POR RAÇA/COR ---


IN_RESERVA_VAGAS,AMPLA_CONCORRENCIA,RESERVA_VAGAS,TOTAL,% COTISTAS
RACA_DESCRICAO,,,,
Parda,5713,8713,14426,60.40
Branca,7167,2430,9597,25.32
Não informado,3823,1251,5074,24.66
Preta,1032,869,1901,45.71
Amarela,144,28,172,16.28
Indígena,28,12,40,30.00


---

##  6. Distribuição de Auxílios por Turno do Curso

Mapeamento do suporte de alimentação e moradia por modalidade de turno escolar (Integral, Noturno, Matutino, Vespertino).

In [22]:
df_turno_aux = (
    df_fato.groupby("TURNO_DESCRICAO")
    .apply(
        lambda x: pd.Series(
            {
                "TOTAL_ESTUDANTES": x["TOTAL_ALUNOS"].sum(),
                "ALIMENTACAO_ALUNOS": (
                    x[x["IN_APOIO_ALIMENTACAO"] == 1]["TOTAL_ALUNOS"].sum()
                    if "IN_APOIO_ALIMENTACAO" in x.columns
                    else 0
                ),
                "MORADIA_ALUNOS": (
                    x[x["IN_APOIO_MORADIA"] == 1]["TOTAL_ALUNOS"].sum()
                    if "IN_APOIO_MORADIA" in x.columns
                    else 0
                ),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

df_turno_aux["% ALIMENTAÇÃO"] = (
    df_turno_aux["ALIMENTACAO_ALUNOS"]
    / df_turno_aux["TOTAL_ESTUDANTES"]
    * 100
).round(2)
df_turno_aux["% MORADIA"] = (
    df_turno_aux["MORADIA_ALUNOS"] / df_turno_aux["TOTAL_ESTUDANTES"] * 100
).round(2)

print("--- CONCESSÃO DE ALIMENTAÇÃO E MORADIA POR TURNO ---")
display(df_turno_aux.sort_values(by="TOTAL_ESTUDANTES", ascending=False))

--- CONCESSÃO DE ALIMENTAÇÃO E MORADIA POR TURNO ---


,TURNO_DESCRICAO,TOTAL_ESTUDANTES,ALIMENTACAO_ALUNOS,MORADIA_ALUNOS,% ALIMENTAÇÃO,% MORADIA
0,Integral,13789,986,142,7.15,1.03
2,Noturno,9084,184,11,2.03,0.12
1,Matutino,4134,139,5,3.36,0.12
4,Vespertino,3025,110,8,3.64,0.26
3,Não informado,1178,0,0,0.00,0.00


---

##  7. Principais Achados (Data Insights)

1. **Ações Afirmativas:** Discentes autodeclarados **pardos** (60,40%) e **pretos** (45,71%) apresentam as maiores proporções relativas de ingresso via reserva de vagas, evidenciando o papel de inclusão socioeconômica da política no campus.
2. **Demandas por Turno:** Cursos em turno **Integral** concentram a maior penetração de auxílio alimentação (7,15%) e moradia (1,03%), refletindo a necessidade de permanência contínua na universidade.
3. **Representatividade de Cursos:** **Pedagogia**, **Ciências Contábeis** e **Direito** lideram em número absoluto de matrículas ativas no Campus I.